In [8]:
import os
import torch
import pandas as pd
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizer, BertModel, AdamW, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Disclaimer some of this code was used for my submission in F20AA CW 2 with my team
# To adhere by academic conduct I give credit to the following people: myself, Aamir Nazir, Francis Sandrino, Ahmed Abdelfattah

# Create a class for classifier
class BERTClassifier(nn.Module):
  def __init__(self):
      super(BERTClassifier, self).__init__()
      self.bert = BertModel.from_pretrained('ProsusAI/finbert') # Load model
      self.dropout = nn.Dropout(0.1) # add 0.1 dropout
      self.fc = nn.Linear(self.bert.config.hidden_size, 3)  # linear transformation to support our number of classes

  def forward(self, input_ids, attention_mask):
      outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask) # defines inputs and attention mask
      po = outputs.pooler_output # creates a pooler output
      x = self.dropout(po) # add dropout
      logits = self.fc(x) # apply linear function on x
      return logits

class TextClassificationDataset(Dataset):
  def __init__(self, texts, labels, tokenizer, max_length):
          self.texts = texts # define input texts
          self.labels = labels # define labels
          self.tokenizer = tokenizer # define tokenizer
          self.max_length = max_length # define max length
  def __len__(self):
      return len(self.texts) # define length of input text
  def __getitem__(self, idx):
      text = str(self.texts[idx]) # get item and make string
      label = self.labels[idx] # get label
      enc = self.tokenizer(text, return_tensors='pt', max_length=self.max_length, padding='max_length', truncation=True) # tokenize the text and get the encoding
      return {'input_ids': enc['input_ids'].flatten(), 'attention_mask': enc['attention_mask'].flatten(), 'label': torch.tensor(label-1)} # we just need the input ids, attention mask and the label

def train(m, dfLoader, optim, scheduler, device):
  m.train() # train the model
  for i,batch in enumerate(dfLoader): # loop through df
      optim.zero_grad() # resets gradients of optimized tensors
      input_ids = batch['input_ids'].to(device) # put the input_ids to cuda for GPU usage
      attention_mask = batch['attention_mask'].to(device) # put the attention mask to cuda for GPU usage
      labels = batch['label'].to(device) # put the labels to cuda for GPU usage
      outputs = m(input_ids, attention_mask) # forward propagate
      loss = nn.CrossEntropyLoss()(outputs, labels) # get loss
      if i % 100 == 0: # print current batch to figure out how long exezution would take
        print(f"Batch: {i}")
      loss.backward() # back propagate
      optim.step() # single optimization step
      scheduler.step() # single scheduler step

def evaluate(m, dfLoader, device):
    m.eval() # evaluate model
    predictions = [] # store predictions and labels
    true_labels = []
    with torch.no_grad(): # Disable gradient calculation to lower computational cost
        for batch in dfLoader: # loop through batches. Due to small dataset we only have 1
            input_ids = batch['input_ids'].to(device) # put the input_ids to cuda for GPU usage
            attention_mask = batch['attention_mask'].to(device) # put the attention mask to cuda for GPU usage
            labels = batch['label'].to(device) # put the attention mask to cuda for GPU usage
            outputs = m(input_ids=input_ids, attention_mask=attention_mask) # forward propagate
            _, preds = torch.max(outputs, dim=1) # get predictions
            predictions.extend(preds.cpu().tolist()) # put predictions back to the cpu
            true_labels.extend(labels.cpu().tolist()) # put labels back to the cpu
    return accuracy_score(true_labels, predictions), classification_report(true_labels, predictions) # Print statistics

def predict_sentiment(text, m, tokenizer, device, max_length=23):
    m.eval() # evaluate model
    encoding = tokenizer(text, return_tensors='pt', max_length=max_length, padding='max_length', truncation=True) # tokenize text
    input_ids = encoding['input_ids'].to(device) # Utilize GPU
    attention_mask = encoding['attention_mask'].to(device) # Utilize GPU

    with torch.no_grad(): # Disable gradient calculation to lower computational cost
            outputs = model(input_ids=input_ids, attention_mask=attention_mask) # forward propagate
            _, preds = torch.max(outputs, dim=1) # get predictions
    return preds.item() # return predictions

In [9]:
df = pd.read_csv('/content/sentiment_annotated_with_texts.csv')
df_t = pd.read_csv('/content/NewsData.csv')
df_testTwo = pd.read_csv('/content/NewsDataMore.csv')

df_test = pd.concat([df_t, df_testTwo], ignore_index = True)
df_test

,Date,Headline,Publisher
0,"Dec 19, 13:59 GMT",EUR/USD to see another leg higher on a clear p...,FXStreet Insights Team
1,"Dec 19, 13:51 GMT",Euro approaches recent highs favoured by a pos...,Guillermo Alcala
2,"Dec 19, 11:49 GMT",EUR/USD: Unlikely to move back above the 1.10 ...,FXStreet Insights Team
3,"Dec 19, 09:04 GMT",EUR/USD can trade above 1.10 during the holida...,FXStreet Insights Team
4,"Dec 19, 06:56 GMT",ECB’s Villeroy: We will not raise interest any...,Dhwani Mehta
...,...,...,...
1433,"Dec 20, 07:24 GMT",EUR/USD likely to trade a 1.02 to 1.12 range o...,FXStreet Insights Team
1434,"Dec 20, 05:43 GMT",EUR/USD Price Analysis: Loses its recovery mom...,Lallalit Srijandorn
1435,"Dec 20, 01:03 GMT",EUR/USD remains capped under the 1.1000 barrie...,Lallalit Srijandorn
1436,"Dec 19, 19:33 GMT",EUR/USD looking for 1.1000 as risk appetite sh...,Joshua Gibson


In [10]:
def encodeSentiment(txt):
  if txt == "Positive":
    return 3
  elif txt == "Negative":
    return 2
  else:
    return 1

df['finalSent'] = df["true_sentiment"].apply(encodeSentiment)
df

,published_at,ticker,true_sentiment,title,author,url,source,text,finbert_sentiment,finbert_sent_score,finalSent
0,2023-01-12 07:47:00,EURCHF,Positive,Euro to benefit from the ECBs pronounced hawki...,FXStreet Insights Team,https://www.fxstreet.com/news/euro-to-benefit-...,FX Street,The Euro was able to appreciate particularly s...,Positive,0.85,3
1,2023-01-12 10:34:00,EURCHF,Positive,EURCHF Trend higher may remain in place – ING,FXStreet Insights Team,https://www.fxstreet.com/news/eur-chf-trend-hi...,FX Street,EUR/CHF yesterday broke above 1.00. Economists...,Positive,0.51,3
2,2023-01-12 11:40:00,EURCHF,Neutral,Does a jump in EURCHF point to a break above 1...,FXStreet Insights Team,https://www.fxstreet.com/news/does-a-jump-in-e...,FX Street,EUR/CHF vaults parity for the first time since...,Neutral,0.37,1
3,2023-01-12 15:32:00,EURCHF,Positive,EURCHF could extend its advance back to levels...,FXStreet Insights Team,https://www.fxstreet.com/news/eur-chf-could-ex...,FX Street,EUR/CHF climbs back above parity. Economists a...,Positive,0.64,3
4,2023-01-13 11:37:00,EURCHF,Positive,EURCHF to head higher towards 10130 and projec...,FXStreet Insights Team,https://www.fxstreet.com/news/eur-chf-to-head-...,FX Street,EUR/CHF has broken out above the sideways rang...,Positive,0.83,3
...,...,...,...,...,...,...,...,...,...,...,...
2286,2023-05-04 13:55:00,GBPUSD,Positive,GBPUSD holds steady near its highest level sin...,Haresh Menghani,https://www.fxstreet.com/news/gbp-usd-holds-st...,FX Street,The GBP/USD pair enters a bullish consolidatio...,Positive,0.63,3
2287,2023-05-04 14:12:00,EURCHF,Negative,EURCHF to break below the 097 level – Credit S...,FXStreet Insights Team,https://www.fxstreet.com/news/eur-chf-to-break...,FX Street,EUR/CHF has turned back lower over the past co...,Positive,0.47,2
2288,2023-05-04 14:17:00,EURUSD,Neutral,EURUSD Shortterm dips likely supported near 10...,FXStreet Insights Team,https://www.fxstreet.com/news/eur-usd-short-te...,FX Street,The European Central Bank (ECB)downshifted to ...,Negative,-0.49,1
2289,2023-05-04 15:00:00,GBPUSD,Positive,GBPUSD Looking for an eventual final leg highe...,FXStreet Insights Team,https://www.fxstreet.com/news/gbp-usd-looking-...,FX Street,Economists at Credit Suisse discuss GBP outloo...,Neutral,0.43,3


In [11]:
def encodeSentiment(txt):
  if txt == "Positive":
    return 3
  elif txt == "Negative":
    return 2
  else:
    return 1

df['finalSent'] = df["true_sentiment"].apply(encodeSentiment)
df

,published_at,ticker,true_sentiment,title,author,url,source,text,finbert_sentiment,finbert_sent_score,finalSent
0,2023-01-12 07:47:00,EURCHF,Positive,Euro to benefit from the ECBs pronounced hawki...,FXStreet Insights Team,https://www.fxstreet.com/news/euro-to-benefit-...,FX Street,The Euro was able to appreciate particularly s...,Positive,0.85,3
1,2023-01-12 10:34:00,EURCHF,Positive,EURCHF Trend higher may remain in place – ING,FXStreet Insights Team,https://www.fxstreet.com/news/eur-chf-trend-hi...,FX Street,EUR/CHF yesterday broke above 1.00. Economists...,Positive,0.51,3
2,2023-01-12 11:40:00,EURCHF,Neutral,Does a jump in EURCHF point to a break above 1...,FXStreet Insights Team,https://www.fxstreet.com/news/does-a-jump-in-e...,FX Street,EUR/CHF vaults parity for the first time since...,Neutral,0.37,1
3,2023-01-12 15:32:00,EURCHF,Positive,EURCHF could extend its advance back to levels...,FXStreet Insights Team,https://www.fxstreet.com/news/eur-chf-could-ex...,FX Street,EUR/CHF climbs back above parity. Economists a...,Positive,0.64,3
4,2023-01-13 11:37:00,EURCHF,Positive,EURCHF to head higher towards 10130 and projec...,FXStreet Insights Team,https://www.fxstreet.com/news/eur-chf-to-head-...,FX Street,EUR/CHF has broken out above the sideways rang...,Positive,0.83,3
...,...,...,...,...,...,...,...,...,...,...,...
2286,2023-05-04 13:55:00,GBPUSD,Positive,GBPUSD holds steady near its highest level sin...,Haresh Menghani,https://www.fxstreet.com/news/gbp-usd-holds-st...,FX Street,The GBP/USD pair enters a bullish consolidatio...,Positive,0.63,3
2287,2023-05-04 14:12:00,EURCHF,Negative,EURCHF to break below the 097 level – Credit S...,FXStreet Insights Team,https://www.fxstreet.com/news/eur-chf-to-break...,FX Street,EUR/CHF has turned back lower over the past co...,Positive,0.47,2
2288,2023-05-04 14:17:00,EURUSD,Neutral,EURUSD Shortterm dips likely supported near 10...,FXStreet Insights Team,https://www.fxstreet.com/news/eur-usd-short-te...,FX Street,The European Central Bank (ECB)downshifted to ...,Negative,-0.49,1
2289,2023-05-04 15:00:00,GBPUSD,Positive,GBPUSD Looking for an eventual final leg highe...,FXStreet Insights Team,https://www.fxstreet.com/news/gbp-usd-looking-...,FX Street,Economists at Credit Suisse discuss GBP outloo...,Neutral,0.43,3


In [13]:
# Load dataframe
df_test['Headline'] = df_test['Headline'].astype(str)
texts = df['title'].tolist()
labels = df['finalSent'].tolist()

# Our specific model
bert_model_name = 'ProsusAI/finbert'
num_classes = 3
max_length = 23
batch_size = 64
num_epochs = 10
learning_rate = 2e-5

# Loading tokenizer and getting the datasets
train_texts, val_texts, train_labels, val_labels = train_test_split(texts, labels, test_size=0.2, random_state=42)
tokenizer = BertTokenizer.from_pretrained(bert_model_name)
train_dataset = TextClassificationDataset(train_texts, train_labels, tokenizer, max_length)
val_dataset = TextClassificationDataset(val_texts, val_labels, tokenizer, max_length)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size)

# set device to cuda and create classifier
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BERTClassifier(bert_model_name, num_classes).to(device)

# create optimizer and scheduler
optim = AdamW(model.parameters(), lr=learning_rate)
total_steps = len(train_dataloader) * num_epochs
scheduler = get_linear_schedule_with_warmup(optim, num_warmup_steps=0, num_training_steps=total_steps)

# Loop through num of epochs specified
for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")
    train(model, train_dataloader, optim, scheduler, device) # train model
    accuracy, report = evaluate(model, val_dataloader, device) # Evaluate model
    print(f"Validation Accuracy: {accuracy:.4f}")
    print(report) # print classification report
    torch.save(model.state_dict(), f"./FXStreetBERTT{epoch}e.pt") # save model
    df_preds = pd.DataFrame()
    df_preds['date'] = df_test['Date'] # we need the dates in the predictions
    for index, row in df_test.iterrows(): # loop through test set
        value = predict_sentiment(row['Headline'], model, tokenizer, device) # make predictions
        df_preds.at[index, 'prediction'] = value # add predictions to df_preds
    df_preds.to_csv(f"./FXStreetBERTT{epoch + 1}Epochs.csv", index = False) # save prediction per epoch to use best model before it was overfit.

/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:429: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1/10
Batch: 0
Validation Accuracy: 0.6013
              precision    recall  f1-score   support

           0       0.58      0.45      0.50       170
           1       0.58      0.73      0.64       140
           2       0.65      0.66      0.65       149

    accuracy                           0.60       459
   macro avg       0.60      0.61      0.60       459
weighted avg       0.60      0.60      0.60       459

Epoch 2/10
Batch: 0
Validation Accuracy: 0.6667
              precision    recall  f1-score   support

           0       0.63      0.65      0.64       170
           1       0.63      0.81      0.71       140
           2       0.78      0.56      0.65       149

    accuracy                           0.67       459
   macro avg       0.68      0.67      0.67       459
weighted avg       0.68      0.67      0.66       459

Epoch 3/10
Batch: 0
Validation Accuracy: 0.7190
              precision    recall  f1-score   support

           0       0.68      0.69      